In [26]:
import pandas as pd
import numpy as np
import pickle

from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.base import BaseEstimator, TransformerMixin

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import mlflow

# Models
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

### Global variables:

In [27]:
# --- Config ---
K_FOLD_SPLITS = 10
RANDOM_STATE = 42
DATA_PATH = '../raw_data/train.csv'

LOWER_PERCENTILE = 0.25
UPPER_PERCENTILE = 0.75

STRATIFIED_K_FOLDS = 5

ID_COL = 'ID_code'
TARGET_COL = 'target'
NON_FEATURES_VARIABLES = [ID_COL, TARGET_COL]

### MLflow settings:

In [28]:
MLFLOW_TRACKING_URI = 'http://127.0.0.1:5000'
EXPERIMENT_NAME = 'Santander-Binary-Classification-Improved'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/345609536713312923', creation_time=1750611046989, experiment_id='345609536713312923', last_update_time=1750611046989, lifecycle_stage='active', name='Santander-Binary-Classification-Improved', tags={}>

### Data Loading:

In [29]:
# --- Load Data ---
df = pd.read_csv(DATA_PATH)
FEATURES_VARIABLES = [col for col in df.columns if col not in NON_FEATURES_VARIABLES]

X = df.drop(columns=NON_FEATURES_VARIABLES)
y = df[TARGET_COL]

### Custom Classes:

In [30]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, features, lower_quantile=0.25, upper_quantile=0.75):
        self.features = features
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        self.lower_bounds_ = {feature: 0 for feature in features}
        self.upper_bounds_ = {feature: 0 for feature in features}
    
    def fit(self, X, y=None):
        # Compute bounds per feature
        for col in self.features:
            Q1 = np.quantile(X[col], self.lower_quantile)
            Q3 = np.quantile(X[col], self.upper_quantile)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.lower_bounds_[col] = lower
            self.upper_bounds_[col] = upper
        return self
    
    def transform(self, X):
        X_capped = X.copy()
        for col in self.features:
            lower = self.lower_bounds_[col]
            upper = self.upper_bounds_[col]
            X_capped[col] = np.clip(X_capped[col], lower, upper)
        return X_capped


### Models and their parameters:

In [31]:
# Models dict with class_weight balanced
MODELS = {
    'LogisticRegression': {
        'model': LogisticRegression,
        'params': {
            'max_iter': 200,
            'solver': 'liblinear',
            'class_weight': 'balanced',
            'random_state': RANDOM_STATE
        }
    },
    'XGBoost': {
        'model': XGBClassifier,
        'params': {
            'n_estimators': 200,
            'learning_rate': 0.1,
            'use_label_encoder': False,
            'eval_metric': 'logloss',
            'random_state': RANDOM_STATE
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier,
        'params': {
            'n_estimators': 200,
            'class_weight': 'balanced',
            'random_state': RANDOM_STATE,
            'bootstrap': False
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier,
        'params': {
            'n_estimators': 200,
            'learning_rate': 0.1,
            'max_depth': 3,
            'random_state': RANDOM_STATE
        }
    },

}

### Model training with StratifiedKFold

In [ ]:
skf = StratifiedKFold(n_splits=STRATIFIED_K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

#Applied to all models:
for model_item in MODELS:
    print(model_item)
    model_params = MODELS[model_item]['params']
    ModelClass = MODELS[model_item]['model']
    model = ModelClass(**model_params)
    
    #For later comparison in MLFlow:
    best_model = None
    best_auc = 0
    model_report = None
    
    #StratifiedKFold for the data:
    fold_idx = 0
    for train_idx, val_idx in skf.split(X, y):
        fold_idx += 1
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        #undersampler = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy=0.8)
        #smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy=1.0)

        
        
        # Pipeline for train data: outlier cap -> scaling -> undersample -> smote
        pipeline = Pipeline([
            ('cap_outliers', ),
            ('scaler', )
        #    ,('undersample', undersampler),
        #    ('smote', smote)
        ])

        
        outlier_capper_train = OutlierCapper(features=FEATURES_VARIABLES)
        outlier_capper_val = OutlierCapper(features=FEATURES_VARIABLES)
        scaler = MinMaxScaler()

        # Preprocessing training data
        outlier_capper_train.fit(X_train)
        X_train_capped = outlier_capper_train.transform(X_train)
        scaler.fit(X_train_capped)
        X_train_scaled = scaler.transform(X_train_capped)
        
                
        # Preprocessing validation data (simple outlier deletion and scaling)
        outlier_capper_val.fit(X_val)
        X_val_capped = outlier_capper_val.transform(X_val)
        X_val_scaled = scaler.transform(X_val_capped)
        
        # Now train model on resampled train data and evaluate on X_val_scaled
        model.fit(X_train_scaled, y_train)
        y_val_pred = model.predict(X_val_scaled)
        y_val_proba = model.predict_proba(X_val_scaled)[:, 1]
        
        # Generate all the metrics to study model's performance
        report = classification_report(y_val, y_val_pred, output_dict=True)
        auc = roc_auc_score(y_val, y_val_proba)
        

        print(f"Fold {fold_idx} -- AUC: {auc:.4f}, Full report {report}")

        #If a better-performing model (based on AUC) has been found, it's updated:
        if auc > best_auc:
            best_auc = auc
            best_model = model
            model_report = report

    
                

LogisticRegression
Fold 1 -- AUC: 0.8623, Full report {'0': {'precision': 0.9698287671232877, 'recall': 0.7870542786470637, 'f1-score': 0.8689341986161612, 'support': 35981.0}, '1': {'precision': 0.29055555555555557, 'recall': 0.7807912416023887, 'f1-score': 0.42351035832377354, 'support': 4019.0}, 'accuracy': 0.786425, 'macro avg': {'precision': 0.6301921613394217, 'recall': 0.7839227601247262, 'f1-score': 0.6462222784699674, 'support': 40000.0}, 'weighted avg': {'precision': 0.9015787911910198, 'recall': 0.7864249999999999, 'f1-score': 0.8241802382627835, 'support': 40000.0}}
Fold 2 -- AUC: 0.8577, Full report {'0': {'precision': 0.9691149036136254, 'recall': 0.7796336955615464, 'f1-score': 0.8641089223281533, 'support': 35981.0}, '1': {'precision': 0.2827030939026597, 'recall': 0.7775566061209256, 'f1-score': 0.41464870961321576, 'support': 4019.0}, 'accuracy': 0.779425, 'macro avg': {'precision': 0.6259089987581425, 'recall': 0.778595150841236, 'f1-score': 0.6393788159706846, 'supp

/home/kisara/PycharmProjects/DataScienceChallenges/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [22:18:56] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 1 -- AUC: 0.8728, Full report {'0': {'precision': 0.9154652351738242, 'recall': 0.9953308690697868, 'f1-score': 0.9537289783092103, 'support': 35981.0}, '1': {'precision': 0.8090909090909091, 'recall': 0.1771584971385917, 'f1-score': 0.2906715656256379, 'support': 4019.0}, 'accuracy': 0.913125, 'macro avg': {'precision': 0.8622780721323666, 'recall': 0.5862446831041893, 'f1-score': 0.6222002719674241, 'support': 40000.0}, 'weighted avg': {'precision': 0.9047772747606434, 'recall': 0.913125, 'f1-score': 0.8871082847698284, 'support': 40000.0}}


/home/kisara/PycharmProjects/DataScienceChallenges/.venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [22:19:01] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fold 2 -- AUC: 0.8764, Full report {'0': {'precision': 0.916029167199693, 'recall': 0.9950529446096551, 'f1-score': 0.9539072283057576, 'support': 35981.0}, '1': {'precision': 0.805464480874317, 'recall': 0.1833789499875591, 'f1-score': 0.29874341305229024, 'support': 4019.0}, 'accuracy': 0.9135, 'macro avg': {'precision': 0.860746824037005, 'recall': 0.5892159472986072, 'f1-score': 0.6263253206790239, 'support': 40000.0}, 'weighted avg': {'precision': 0.9049201803411508, 'recall': 0.9135, 'f1-score': 0.8880796439681653, 'support': 40000.0}}
